# PETase zero-shot clustering with ESM2

Run the setup cell first. This exploratory notebook contains alternative analysis branches and optional MAFFT/DIAMOND/UniProt steps; use the sections you need. External search databases are not bundled. New files are written to `outputs/classification/`; archived results are left intact.

In [ ]:
# Locate this repository and stage inputs in outputs/classification.
from pathlib import Path
import os
import runpy

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/prepare_classification.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter from this repository or its notebooks directory.")
prepare = runpy.run_path(str(REPO_ROOT / "scripts/prepare_classification.py"))["prepare"]
RUN_DIR = prepare()
os.chdir(RUN_DIR)
print("Notebook working directory:", RUN_DIR)


In [ ]:
# Import packages
import os
import sys
import time
import math

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import umap
import hdbscan
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

import esm
import re
import hdbscan
from pathlib import Path
from Bio import pairwise2
%matplotlib inline

# Embedding

In [ ]:
# Configuration
csv_path = "predictive-pet-zero-shot-test-2025.csv"
batch_size = 16

In [ ]:
# Read sequence CSV
df = pd.read_csv(csv_path)
seq_col = "sequence"
sequences = df[seq_col].astype(str).tolist()

In [ ]:
len(sequences)

In [ ]:
# Load ESM2 model 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Device: {device}")

Using CPU could becuase of WSL....no need to fix at this moment 

In [ ]:
print(f"[INFO] load ESM2 model esm2_t30_150M_UR50D")
model, alphabet = esm.pretrained.esm2_t30_150M_UR50D()
model = model.to(device)
model.eval()
batch_converter = alphabet.get_batch_converter()

In [ ]:
# Compute embeddings
print("[INFO] strat to get sequence embedding ")
all_embeddings = []
num_seqs = len(sequences) #4988
t0 = time.time() # lest print the time for fun (?)

for start in range(0, num_seqs, batch_size):
    end = min(start + batch_size, num_seqs) 
    batch_seqs = sequences[start:end] # process $batch_size at a time 

    batch_data = [(f"seq_{i}", s) for i, s in enumerate(batch_seqs)] 
    labels_batch, strs, tokens = batch_converter(batch_data) # ESM2 need tuple, can convert using the batch_converter (ESM's tokenizer)
    tokens = tokens.to(device) # with the shape (B, L), B as batch size, L is the max length of each sequence after padding 

    with torch.no_grad():
        out = model(tokens, repr_layers=[12], return_contacts=False) # ESM2 have 12 layers, so using the last layer as the final embedding
        rep = out["representations"][12]  # shape: (B, L, D), D as 480 i think, for esm2_t2_35m

    # mean pooling over sequence length
    # emb = rep[:, 1:, :].mean(1).cpu().numpy()  # (B, D), using 1 here because we dont want to keep the <cls> token, mean(1) is to get the avg for each sequence, hence for each sequence we have a 1xD array
    emb = rep[:, 0, :].cpu().numpy()   # (B, D) # update to using CLS token pooling 
    all_embeddings.append(emb)
    if (start // batch_size) % 5 == 0:
        elapsed = time.time() - t0
        print(f"  [INFO] Already processed {end}/{num_seqs} sequences，in {elapsed:.1f} seconds")


embeddings = np.vstack(all_embeddings)
print(f"[INFO] embedding shape: {embeddings.shape}")

# Clustering

In [ ]:
# Config
umap_n_neighbors = 80
umap_min_dist = 0.05
hdbscan_min_cluster_size = 75

In [ ]:
# Normalize embedding before UMAP and HDBSCAN
X_scaled = StandardScaler().fit_transform(embeddings)

In [ ]:
# Using UMAP reduce dimention
reducer = umap.UMAP(
    n_neighbors=umap_n_neighbors,
    min_dist=umap_min_dist,
    metric="cosine",
    random_state=42,
)
X_umap = reducer.fit_transform(X_scaled)
print("UMAP -> 2D")

In [ ]:
# HDBSCAN clustering
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=hdbscan_min_cluster_size,
    min_samples=20,
    metric="euclidean"
)
labels = clusterer.fit_predict(X_umap)
print("[INFO] HDBSCAN")
print(f"       in total, we got {len(set(labels))} cluster (include -1)")

In [ ]:
# Save results
df['cluster'] = labels
df['umap_x'] = X_umap[:, 0]
df['umap_y'] = X_umap[:, 1]

out_csv = "clustered_sequences.csv"
df.to_csv(out_csv, index=False)
print(f"[INFO] output -> {out_csv}")

In [ ]:
# Plot UMAP clusters
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=labels,
    cmap="tab20",
    s=8,
    alpha=0.8,
)
plt.colorbar(scatter, label="Cluster ID")
plt.title("PETase Zero-shot Sequences - UMAP + HDBSCAN")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.tight_layout()

fig_path = "clusters_umap_5_cluster.png"
plt.savefig(fig_path, dpi=300)
print(f"[INFO] output -> {fig_path}")
plt.show()

In [ ]:
df["cluster"].value_counts().sort_index()

In [ ]:
# Check for each cluster 
df = pd.read_csv("clustered_sequences.csv")

for cid in sorted(df["cluster"].unique()):
    sub = df[df["cluster"] == cid]
    sub.to_csv(f"cluster_{cid}.csv", index=False)
    print(f"saved cluster {cid} → {len(sub)} sequences")


## Check each cluster

In [ ]:
cluster_1 = df[df["cluster"] == 1]

In [ ]:
cluster_1[["sequence", "umap_x", "umap_y"]].head(20)

In [ ]:
X = cluster_1[["umap_x", "umap_y"]].values
centroid = X.mean(axis=0)
print("Cluster 1 centroid:", centroid)

dist = np.linalg.norm(X - centroid, axis=1)
rep_seq = cluster_1.iloc[dist.argmin()]
print("Cluster 1 sequence that is most close to the centroid：")
print(rep_seq["sequence"])

In [ ]:
clusters = sorted(df["cluster"].unique())
print("[INFO] Found clusters:", clusters)

rep_rows = []

for cid in clusters:
    sub = df[df["cluster"] == cid].reset_index(drop=True)
    X = sub[["umap_x", "umap_y"]].values
    
    if len(sub) == 0:
        print(f"[WARN] Cluster {cid} empty, skip.")
        continue
    
    # Compute centroid
    centroid = X.mean(axis=0)
    
    # Compute distances to centroid
    dist = np.linalg.norm(X - centroid, axis=1)
    rep_idx = dist.argmin()
    rep_seq = sub.loc[rep_idx, "sequence"]
    
    print("\n======================================")
    print(f"Cluster {cid}")
    print("Centroid:", centroid)
    print("Representative sequence index:", rep_idx)
    print("Representative sequence:")
    print(rep_seq)
    
    rep_rows.append({
        "cluster": cid,
        "n_sequences": len(sub),
        "centroid_x": centroid[0],
        "centroid_y": centroid[1],
        "rep_seq_index": rep_idx,
        "rep_sequence": rep_seq
    })

# Save summary table
rep_df = pd.DataFrame(rep_rows)
rep_df.to_csv("cluster_representative_sequences.csv", index=False)

print("\n[INFO] Summary saved to: cluster_representative_sequences.csv")
rep_df

### Get frequency - test

Run `mafft --auto cluster_1.fasta > cluster_1_aln.fasta` at terminal (need to install mafft first) using the fasta data

In [ ]:
from collections import Counter
from Bio import SeqIO
import logomaker

In [ ]:
aln_path = "MSA/cluster_4_aln.fasta"

records = list(SeqIO.parse(aln_path, "fasta"))
L = len(records[0].seq)
L

In [ ]:
aln_array = np.array([list(str(rec.seq)) for rec in records])  # shape: (N, L)

In [ ]:
consensus = []
conservation = []

for j in range(L):
    col = aln_array[:, j]
    # remove gap
    col_aa = [aa for aa in col if aa != '-']
    if not col_aa:
        consensus.append('-')
        conservation.append(0.0)
        continue
    
    counter = Counter(col_aa)
    aa, cnt = counter.most_common(1)[0]
    consensus.append(aa)
    conservation.append(cnt / len(col_aa))

consensus_seq = "".join(consensus)
print("Consensus top 80：")
print(consensus_seq[:80])
print("avg conservation:", np.mean(conservation))

In [ ]:
seqs = [str(rec.seq) for rec in records]
def alignment_to_pfm(seqs, start=None, end=None):
    if start is None: start = 0
    if end is None: end = len(seqs[0])
    sub_seqs = [s[start:end] for s in seqs]
    L = end - start
    aas = list("ACDEFGHIKLMNPQRSTVWY")
    counts = {aa: [0]*L for aa in aas}
    
    for s in sub_seqs:
        for i, aa in enumerate(s):
            if aa in counts:
                counts[aa][i] += 1
    df = pd.DataFrame(counts)
    return df

pfm = alignment_to_pfm(seqs, start=0, end=100)
ppm = pfm.div(pfm.sum(axis=1), axis=0)

plt.figure(figsize=(100, 3))
logo = logomaker.Logo(ppm, shade_below=.5, fade_below=.5, font_name='DejaVu Sans')
plt.xlabel("Alignment position (40–100)")
plt.ylabel("Frequency")
plt.title("Cluster 4 sequence logo (positions 0–100)")
plt.tight_layout()
plt.savefig("MSA/cluster_4_logo_0_100.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(100, 3))
logo = logomaker.Logo(ppm, shade_below=.5, fade_below=.5, font_name='DejaVu Sans')
plt.xlabel("Alignment position (100–200)")
plt.ylabel("Frequency")
plt.title("Cluster 4 sequence logo (positions 100–200)")
plt.tight_layout()
plt.savefig("MSA/cluster_4_logo_100_200.png", dpi=300)
plt.show()

In [ ]:
# Check if any of them dont have GXSXG

In [ ]:
import re

In [ ]:
records = list(SeqIO.parse("MSA/cluster_2.fasta", "fasta"))

pattern = re.compile(r"G.S.G")  

for rec in records:
    seq = str(rec.seq)
    m = pattern.search(seq)
    if m:
        print(rec.id, "motif at", m.start(), ":", m.group())
    else:
        print(rec.id, "NO motif found")

### Get frequency and check motif

In [ ]:
import subprocess
import re

In [ ]:
csv_path = "clustered_sequences.csv"
msa_dir = "MSA"  
seq_col = "sequence"
cluster_col = "cluster" 
mafft_exe = "mafft"   

In [ ]:
df = pd.read_csv(csv_path)
clusters = sorted(df[cluster_col].unique())

motif_pattern = re.compile(r"G.S.G")

In [ ]:
summary_rows = []

for cid in clusters:
    sub = df[df[cluster_col] == cid].reset_index(drop=True)
    n_seq = len(sub)
    if n_seq < 2:
        print(f"\n[WARN] Cluster {cid}: only {n_seq} sequences, skip MSA")
        continue

    print(f"\n[INFO] Processing cluster {cid} ({n_seq} sequences) ...")

    # ---------- 1) Export FASTA ----------
    fasta_path = os.path.join(msa_dir, f"cluster_{cid}.fasta")
    with open(fasta_path, "w") as f:
        for i, seq in enumerate(sub[seq_col]):
            seq = str(seq).strip().replace("*", "")
            f.write(f">cluster{cid}_{i}\n{seq}\n")
    print(f"  [INFO] FASTA written: {fasta_path}")

    # ---------- 2) Run MAFFT alignment ----------
    aln_path = os.path.join(msa_dir, f"cluster_{cid}_aln.fasta")
    print(f"  [INFO] Running MAFFT ...")
    with open(aln_path, "w") as fout:
        subprocess.run([mafft_exe, "--auto", fasta_path],
                       stdout=fout, stderr=subprocess.DEVNULL, check=True)
    print(f"  [INFO] Alignment saved: {aln_path}")

    # ---------- 3) Compute consensus & conservation ----------
    records_aln = list(SeqIO.parse(aln_path, "fasta"))
    if not records_aln:
        print(f"  [WARN] Alignment empty, skipping cluster {cid}")
        continue

    L = len(records_aln[0].seq)
    aln_array = np.array([list(str(rec.seq)) for rec in records_aln])

    consensus = []
    conservation = []

    for j in range(L):
        col = aln_array[:, j]
        col_aa = [aa for aa in col if aa != "-"]
        if not col_aa:
            consensus.append("-")
            conservation.append(0.0)
            continue

        counter = Counter(col_aa)
        aa, cnt = counter.most_common(1)[0]
        consensus.append(aa)
        conservation.append(cnt / len(col_aa))

    consensus_seq = "".join(consensus)
    avg_cons = float(np.mean(conservation))

    print("  [INFO] Consensus first 80 positions:")
    print("  ", consensus_seq[:80])
    print(f"  [INFO] Average conservation: {avg_cons:.3f}")

    # ---------- 4) Scan raw sequences for GXSXG motif ----------
    records_raw = list(SeqIO.parse(fasta_path, "fasta"))
    motif_hits = 0
    motif_positions = []

    for rec in records_raw:
        s = str(rec.seq)
        m = motif_pattern.search(s)
        if m:
            motif_hits += 1
            motif_positions.append(m.start())

    frac_motif = motif_hits / len(records_raw)
    if motif_hits > 0:
        pos_min = min(motif_positions)
        pos_max = max(motif_positions)
        pos_str = f"{pos_min}–{pos_max}"
    else:
        pos_str = "N/A"

    print(f"  [INFO] GXSXG motif coverage: {motif_hits}/{len(records_raw)} "
          f"({frac_motif:.2%}), position range: {pos_str}")

    summary_rows.append({
        "cluster": cid,
        "n_seq": n_seq,
        "avg_conservation": avg_cons,
        "motif_fraction": frac_motif,
        "motif_pos_range": pos_str,
        "consensus_head80": consensus_seq[:80],
    })

# ---------- 5) Save summary table ----------
summary_df = pd.DataFrame(summary_rows).sort_values("cluster")
summary_csv = os.path.join(msa_dir, "cluster_motif_summary.csv")
summary_df.to_csv(summary_csv, index=False)

print("\n[INFO] All clusters processed. Summary saved to:")
print(" ", summary_csv)

summary_df

# Analysis 

## 1

In [ ]:
from itertools import product

In [ ]:
df = pd.read_csv("clustered_sequences.csv")
clusters = sorted(df["cluster"].unique())
print("[INFO] Clusters:", clusters)

In [ ]:
msa_summary_path = "MSA/cluster_motif_summary.csv"
if os.path.exists(msa_summary_path):
    msa_summary = pd.read_csv(msa_summary_path)
    print("[INFO] Loaded MSA summary:")
    display(msa_summary)
else:
    msa_summary = None
    print("[WARN] MSA summary not found at", msa_summary_path)

In [ ]:
# compute centroids in UMAP space ---
centroids = {}
for cid in clusters:
    sub = df[df["cluster"] == cid]
    X = sub[["umap_x", "umap_y"]].values
    centroids[cid] = X.mean(axis=0)

# compute consensus_head80 and avg length for each cluster ---
cons_head80 = {}
mean_len = {}

for cid in clusters:
    # consensus from MSA summary if present
    if msa_summary is not None and cid in msa_summary["cluster"].values:
        row = msa_summary[msa_summary["cluster"] == cid].iloc[0]
        cons_head80[cid] = str(row["consensus_head80"])
    else:
        cons_head80[cid] = ""

    # mean length from raw sequences
    sub = df[df["cluster"] == cid]
    mean_len[cid] = sub["sequence"].str.len().mean()

def seq_identity(a, b):
    """Simple sequence identity between two strings, ignoring '-'."""
    a = a.replace("-", "")
    b = b.replace("-", "")
    if not a or not b:
        return np.nan
    L = min(len(a), len(b))
    if L == 0:
        return np.nan
    a = a[:L]
    b = b[:L]
    matches = sum(aa == bb for aa, bb in zip(a, b))
    return matches / L

# Build matrices ---
cluster_list = clusters  # sorted
n = len(cluster_list)

centroid_dist = np.zeros((n, n))
cons_identity = np.zeros((n, n))
length_diff = np.zeros((n, n))

for i, ci in enumerate(cluster_list):
    for j, cj in enumerate(cluster_list):
        # Centroid distance in UMAP space
        d = np.linalg.norm(centroids[ci] - centroids[cj])
        centroid_dist[i, j] = d

        # Consensus similarity (head80)
        si = seq_identity(cons_head80.get(ci, ""), cons_head80.get(cj, ""))
        cons_identity[i, j] = si

        # Mean length difference
        ld = abs(mean_len[ci] - mean_len[cj])
        length_diff[i, j] = ld

# Build DataFrames for easy viewing
centroid_dist_df = pd.DataFrame(centroid_dist, index=cluster_list, columns=cluster_list)
cons_identity_df = pd.DataFrame(cons_identity, index=cluster_list, columns=cluster_list)
length_diff_df = pd.DataFrame(length_diff, index=cluster_list, columns=cluster_list)

print("=== Centroid distance (UMAP space) ===")
display(centroid_dist_df)

print("=== Consensus identity (head80) ===")
display(cons_identity_df)

print("=== Mean sequence length difference ===")
display(length_diff_df)


conclution:
- cluster 3 and 4 basically same
- cluster 2 are far from others
- length difference are basically same for 0 1 3 4, cluster 2 have a ~10-13aa length diffrence


## 2

In [ ]:
import logomaker
from Bio.Phylo.TreeConstruction import DistanceTreeConstructor, DistanceMatrix
from Bio import Phylo

In [ ]:
# Load clustering result
df = pd.read_csv("clustered_sequences.csv")
clusters = sorted(df["cluster"].unique())
print("[INFO] Clusters:", clusters)
print("[INFO] Total sequences:", len(df))

# Load MSA summary (generated earlier)
msa_summary_path = "MSA/cluster_motif_summary.csv"
if os.path.exists(msa_summary_path):
    msa_summary = pd.read_csv(msa_summary_path)
    print("[INFO] Loaded MSA summary:")
    display(msa_summary)
else:
    msa_summary = None
    print("[WARN] MSA summary not found at", msa_summary_path)

# Precompute UMAP centroids for each cluster
centroids = {}
for cid in clusters:
    sub = df[df["cluster"] == cid]
    X = sub[["umap_x", "umap_y"]].values
    centroids[cid] = X.mean(axis=0)

centroids

In [ ]:
def alignment_to_ppm(aln_path, start=40, end=100):
    """Convert an aligned FASTA to a position probability matrix (AA x position)."""
    records = list(SeqIO.parse(aln_path, "fasta"))
    if not records:
        return None
    seqs = [str(rec.seq) for rec in records]
    L = len(seqs[0])
    if end > L:
        end = L
    sub_seqs = [s[start:end] for s in seqs]

    aas = list("ACDEFGHIKLMNPQRSTVWY")
    counts = {aa: [0]*(end-start) for aa in aas}

    for s in sub_seqs:
        for i, aa in enumerate(s):
            if aa in counts:
                counts[aa][i] += 1

    df_counts = pd.DataFrame(counts)
    # convert to probabilities
    df_ppm = df_counts.div(df_counts.sum(axis=1), axis=0).fillna(0.0)
    return df_ppm

logo_dir = "logos"
os.makedirs(logo_dir, exist_ok=True)

In [ ]:
for cid in clusters:
    aln_path = os.path.join("MSA", f"cluster_{cid}_aln.fasta")
    if not os.path.exists(aln_path):
        print(f"[WARN] Alignment for cluster {cid} not found at {aln_path}, skip.")
        continue
    
    print(f"[INFO] Building logo for cluster {cid} from {aln_path} ...")
    ppm = alignment_to_ppm(aln_path, start=40, end=100)
    if ppm is None:
        print(f"[WARN] Empty alignment for cluster {cid}, skip.")
        continue

    plt.figure(figsize=(10, 3))
    logomaker.Logo(ppm, shade_below=.5, fade_below=.5, font_name='DejaVu Sans')
    plt.xlabel("Alignment position (40–100)")
    plt.ylabel("Frequency")
    plt.title(f"Cluster {cid} sequence logo (positions 40–100)")
    plt.tight_layout()
    out_png = os.path.join(logo_dir, f"cluster_{cid}_logo_40_100.png")
    plt.savefig(out_png, dpi=300)
    plt.show()
    print("  [INFO] Saved logo to:", out_png)

In [ ]:
for cid in clusters:
    aln_path = os.path.join("MSA", f"cluster_{cid}_aln.fasta")
    if not os.path.exists(aln_path):
        print(f"[WARN] Alignment for cluster {cid} not found at {aln_path}, skip.")
        continue
    
    print(f"[INFO] Building logo for cluster {cid} from {aln_path} ...")
    ppm = alignment_to_ppm(aln_path, start=100, end=200)
    if ppm is None:
        print(f"[WARN] Empty alignment for cluster {cid}, skip.")
        continue

    plt.figure(figsize=(10, 3))
    logomaker.Logo(ppm, shade_below=.5, fade_below=.5, font_name='DejaVu Sans')
    plt.xlabel("Alignment position (100–200)")
    plt.ylabel("Frequency")
    plt.title(f"Cluster {cid} sequence logo (positions 100–200)")
    plt.tight_layout()
    out_png = os.path.join(logo_dir, f"cluster_{cid}_logo_100_200.png")
    plt.savefig(out_png, dpi=300)
    plt.show()
    print("  [INFO] Saved logo to:", out_png)

In [ ]:
for cid in clusters:
    aln_path = os.path.join("MSA", f"cluster_{cid}_aln.fasta")
    if not os.path.exists(aln_path):
        print(f"[WARN] Alignment for cluster {cid} not found at {aln_path}, skip.")
        continue
    
    print(f"[INFO] Building logo for cluster {cid} from {aln_path} ...")
    ppm = alignment_to_ppm(aln_path, start=200, end=300)
    if ppm is None:
        print(f"[WARN] Empty alignment for cluster {cid}, skip.")
        continue

    plt.figure(figsize=(10, 3))
    logomaker.Logo(ppm, shade_below=.5, fade_below=.5, font_name='DejaVu Sans')
    plt.xlabel("Alignment position (200–300)")
    plt.ylabel("Frequency")
    plt.title(f"Cluster {cid} sequence logo (positions 200–300)")
    plt.tight_layout()
    out_png = os.path.join(logo_dir, f"cluster_{cid}_logo_200_300.png")
    plt.savefig(out_png, dpi=300)
    plt.show()
    print("  [INFO] Saved logo to:", out_png)

In [ ]:
cid = 2
sub = df[df["cluster"] == cid]
lengths = sub["sequence"].str.len().values
plt.figure(figsize=(5,4))
plt.hist(lengths, bins=20, alpha=0.7)
plt.xlabel("Sequence length (aa)")
plt.ylabel("Count")
plt.title(f"Sequence length distribution (cluster {cid})")
plt.tight_layout()
plt.show()

print("Mean length:", lengths.mean())
print("Std length:", lengths.std())


In [ ]:
# Define family mapping
def map_family(c):
    if c == 0:
        return "A_core"
    elif c == 1:
        return "B_core"
    elif c in [3, 4]:
        return "C_core"
    elif c == 2:
        return "D_divergent"
    else:
        return "Other"

df["family_group"] = df["cluster"].apply(map_family)
family_groups = sorted(df["family_group"].unique())
print("[INFO] Family groups:", family_groups)

# Compute family-level centroid in UMAP space
family_centroids = {}
for fg in family_groups:
    sub = df[df["family_group"] == fg]
    if len(sub) == 0:
        continue
    X = sub[["umap_x", "umap_y"]].values
    family_centroids[fg] = X.mean(axis=0)

family_centroids

# Select top-K representative sequences per family
top_k = 5
rep_rows = []

for fg in family_groups:
    sub = df[df["family_group"] == fg].reset_index(drop=True)
    if len(sub) == 0:
        continue

    X = sub[["umap_x", "umap_y"]].values
    centroid = family_centroids[fg]
    dist = np.linalg.norm(X - centroid, axis=1)
    order = np.argsort(dist)

    print(f"\n=== Family {fg} (n={len(sub)}) ===")
    for rank in range(min(top_k, len(sub))):
        idx = order[rank]
        row = sub.loc[idx]
        print(f"\n[Rank {rank+1}] distance={dist[idx]:.3f}")
        print("cluster:", row["cluster"])
        print("UMAP:", (row["umap_x"], row["umap_y"]))
        print("sequence:")
        print(row["sequence"])

        rep_rows.append({
            "family_group": fg,
            "rank": rank+1,
            "distance_to_family_centroid": dist[idx],
            "cluster": row["cluster"],
            "umap_x": row["umap_x"],
            "umap_y": row["umap_y"],
            "sequence": row["sequence"],
        })

rep_df = pd.DataFrame(rep_rows)
rep_df.to_csv("family_representative_sequences_top5.csv", index=False)
print("\n[INFO] Saved representative sequences to family_representative_sequences_top5.csv")
rep_df


In [ ]:


# Helper: simple sequence identity
def seq_identity(a, b):
    a = str(a).replace("-", "")
    b = str(b).replace("-", "")
    if not a or not b:
        return 0.0
    L = min(len(a), len(b))
    if L == 0:
        return 0.0
    a = a[:L]
    b = b[:L]
    matches = sum(aa == bb for aa, bb in zip(a, b))
    return matches / L

# Build a consensus for each family using the largest cluster within that family
family_consensus = {}

if msa_summary is None:
    print("[WARN] No MSA summary loaded; cannot build family consensus.")
else:
    for fg in family_groups:
        # Pick the largest cluster belonging to this family
        clusters_in_fg = (
            df[df["family_group"] == fg]
            .groupby("cluster")
            .size()
            .sort_values(ascending=False)
            .index
            .tolist()
        )
        if not clusters_in_fg:
            continue

        main_cluster = clusters_in_fg[0]
        row = msa_summary[msa_summary["cluster"] == main_cluster]
        if len(row) == 0:
            continue
        cons = row.iloc[0]["consensus_head80"]
        family_consensus[fg] = cons

print("[INFO] Family consensus (head80):")
for fg, cons in family_consensus.items():
    print(f"  {fg}: {cons[:40]}...")

# Build distance matrix between families: d = 1 - identity
family_labels = list(family_consensus.keys())
k = len(family_labels)
dm = []

for i in range(k):
    row = []
    for j in range(i+1):
        if i == j:
            row.append(0.0)
        else:
            idt = seq_identity(family_consensus[family_labels[i]],
                               family_consensus[family_labels[j]])
            row.append(1.0 - idt)
    dm.append(row)

dist_matrix = DistanceMatrix(names=family_labels, matrix=dm)

constructor = DistanceTreeConstructor()
tree = constructor.nj(dist_matrix)

# Save and plot tree
Phylo.write(tree, "family_tree.nwk", "newick")
print("[INFO] Saved family tree to family_tree.nwk")

plt.figure(figsize=(6, 4))
Phylo.draw(tree, do_show=False)
plt.title("Family-level NJ tree (based on consensus_head80)")
plt.tight_layout()
plt.savefig("family_tree.png", dpi=300)
plt.show()


## 3

In [ ]:
rep_path = "family_representative_sequences_top5.csv"  # from previous step
df_rep = pd.read_csv(rep_path)
df_rep = df_rep.sort_values(["family_group", "rank"])

# pick rank 1 representative for each family
family_best = df_rep[df_rep["rank"] == 1].reset_index(drop=True)
print(family_best[["family_group", "cluster", "umap_x", "umap_y"]])

# write to FASTA
os.makedirs("Phylo", exist_ok=True)
fam_fasta = "Phylo/family_reps_full.fasta"

with open(fam_fasta, "w") as f:
    for _, row in family_best.iterrows():
        fg = row["family_group"]
        seq = row["sequence"].replace("*", "").strip()
        f.write(f">{fg}\n{seq}\n")

print("[INFO] FASTA written:", fam_fasta)

# run MAFFT
aln_path = "Phylo/family_reps_full_aln.fasta"
with open(aln_path, "w") as fout:
    subprocess.run(["mafft", "--auto", fam_fasta],
                   stdout=fout, stderr=subprocess.DEVNULL, check=True)
print("[INFO] Alignment saved:", aln_path)

In [ ]:
records = list(SeqIO.parse(aln_path, "fasta"))
names = [rec.id for rec in records]
seqs = [str(rec.seq) for rec in records]

def seq_identity(a, b):
    a = a.replace("-", "")
    b = b.replace("-", "")
    if not a or not b:
        return 0.0
    L = min(len(a), len(b))
    a = a[:L]
    b = b[:L]
    matches = sum(aa == bb for aa, bb in zip(a, b))
    return matches / L

# build lower-triangular distance matrix: d = 1 - identity
k = len(names)
mat = []
for i in range(k):
    row = []
    for j in range(i+1):
        if i == j:
            row.append(0.0)
        else:
            idt = seq_identity(seqs[i], seqs[j])
            row.append(1.0 - idt)
    mat.append(row)

dist_matrix = DistanceMatrix(names=names, matrix=mat)
constructor = DistanceTreeConstructor()
tree = constructor.nj(dist_matrix)

os.makedirs("Phylo", exist_ok=True)
Phylo.write(tree, "Phylo/family_tree_full.nwk", "newick")
print("[INFO] Saved Newick tree to Phylo/family_tree_full.nwk")

plt.figure(figsize=(6, 4))
Phylo.draw(tree, do_show=False)
plt.title("Family-level NJ tree (full-length representative sequences)")
plt.tight_layout()
plt.savefig("Phylo/family_tree_full.png", dpi=300)
plt.show()

In [ ]:
clusters = sorted(df["cluster"].unique())
motif_pattern = "G.S.G"  # GXSXG
import re
motif_re = re.compile(motif_pattern)

motif_stats = []

for cid in clusters:
    fasta_path = os.path.join("MSA", f"cluster_{cid}.fasta")
    if not os.path.exists(fasta_path):
        print("[WARN] no fasta for cluster", cid)
        continue
    records = list(SeqIO.parse(fasta_path, "fasta"))
    positions = []
    for rec in records:
        seq = str(rec.seq)
        m = motif_re.search(seq)
        if m:
            positions.append(m.start())
    if positions:
        motif_stats.append({
            "cluster": cid,
            "n_seq": len(records),
            "n_motif": len(positions),
            "frac_motif": len(positions)/len(records),
            "pos_mean": np.mean(positions),
            "pos_std": np.std(positions),
        })

motif_df = pd.DataFrame(motif_stats)
motif_df


## 4

In [ ]:
df = pd.read_csv("clustered_sequences.csv")
seq_col = "sequence"

fasta_path = "all_queries.fasta"
with open(fasta_path, "w") as f:
    for i, seq in enumerate(df[seq_col]):
        seq = str(seq).strip().replace("*", "")
        f.write(f">seq_{i}\n{seq}\n")

print("FASTA written:", fasta_path, "with", len(df), "sequences")

wget https://rest.uniprot.org/uniprotkb/stream?compressed=true&format=fasta&query=reviewed:true -O uniprot_sprot.fasta.gz
gunzip uniprot_sprot.fasta.gz

conda install -c bioconda diamond

diamond makedb \
    --in uniprot_sprot.fasta \
    -d uniprot_sprot

diamond blastp \
    -q all_queries.fasta \
    -d uniprot_sprot \
    -o uniprot_hits.tsv \
    -f 6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore \
    -k 5 \
    --evalue 1e-5

In [ ]:
df = pd.read_csv("clustered_sequences.csv")

# DIAMOND hits
cols = ["qseqid", "sseqid", "pident", "length", "mismatch", "gapopen",
        "qstart", "qend", "sstart", "send", "evalue", "bitscore"]
hits = pd.read_csv(
    "search/uniprot_hits.tsv", 
    sep="\t", 
    names=cols
)

hits_best = hits.sort_values("bitscore", ascending=False).groupby("qseqid").head(1)

hits_best["idx"] = hits_best["qseqid"].str.replace("seq_", "").astype(int)

df["idx"] = df.index
df_annot = df.merge(hits_best, how="left", on="idx")

df_annot.to_csv("clustered_sequences_with_uniprot.csv", index=False)
print("Saved annotated table to clustered_sequences_with_uniprot.csv")

- qseqid: Query sequence ID
- sseqid: subject sequence ID (uniport accession)
- pident: percent identity
- mismatch: how many aa are mismatched
- qstart: query alighment start position
- qend: query aligment end position
- sstart subject alignment start psition
- send: subject alighment end pisition
- evalue: < 1e-5 makes sense
- bitscore: higher is better
- coverage = alignment_length / query_length


In [ ]:
# 1. Load your original sequence table
df = pd.read_csv("clustered_sequences.csv")   # your main table
df["idx"] = df.index                          # to merge later

# 2. Load DIAMOND hits (top 5 hits usually)
cols = [
    "qseqid", "sseqid", "pident", "length",
    "mismatch", "gapopen", "qstart", "qend",
    "sstart", "send", "evalue", "bitscore"
]

hits = pd.read_csv(
    "search/uniprot_hits.tsv",
    sep="\t",
    names=cols
)

# Extract numeric order from qseqid (e.g., "seq_123")
hits["idx"] = hits["qseqid"].str.replace("seq_", "").astype(int)

# 3. Compute alignment coverage for each hit
# Query sequence lengths
seq_lengths = df["sequence"].str.len().to_dict()   # {idx: length}

def compute_cov(row):
    qlen = seq_lengths.get(row["idx"], None)
    if qlen is None or qlen == 0:
        return np.nan
    return row["length"] / qlen     # alignment length / full query length

hits["coverage"] = hits.apply(compute_cov, axis=1)

# 4. Select best hit for each sequence (highest bitscore)
hits_best = hits.sort_values("bitscore", ascending=False).groupby("idx").head(1)

# 5. Define filtering thresholds
PID_MIN = 40.0       # percent identity ≥ 40%
EVALUE_MAX = 1e-5    # ≤ 1e-5
COV_MIN = 0.60       # coverage ≥ 60%

# Check if hit is reliable
def is_reliable(row):
    if row["pident"] < PID_MIN:
        return False
    if row["evalue"] > EVALUE_MAX:
        return False
    if row["coverage"] < COV_MIN:
        return False
    return True

hits_best["reliable"] = hits_best.apply(is_reliable, axis=1)

# 6. Merge back into original table
df_ann = df.merge(hits_best, how="left", on="idx")

df_ann.to_csv("clustered_sequences_with_uniprot.csv", index=False)
print("[INFO] Saved annotated table to clustered_sequences_with_uniprot.csv")

df_ann.head()


wget "https://ftp.uniprot.org/pub/databases/uniprot/uniref/uniref90/uniref90.fasta.gz"
gunzip uniref90.fasta.gz
diamond makedb --in uniref90.fasta -d uniref90
diamond blastp \
  -q all_queries.fasta \
  -d uniref90 \
  -o uniref90_hits.tsv \
  -f 6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore \
  -k 5 \
  --evalue 1e-5


In [ ]:
# Load your clustered sequence table
df = pd.read_csv("clustered_sequences.csv")

# Ensure we have an index id for each sequence
df["idx"] = df.index

fasta_path = "all_queries.fasta"
with open(fasta_path, "w") as f:
    for i, seq in enumerate(df["sequence"]):
        seq = str(seq).strip().replace("*", "")
        f.write(f">seq_{i}\n{seq}\n")

print("[INFO] FASTA written:", fasta_path, "with", len(df), "sequences")


In [ ]:
df = pd.read_csv("clustered_sequences.csv")
df["idx"] = df.index  # for merging
cols = [
    "qseqid", "sseqid", "pident", "length",
    "mismatch", "gapopen", "qstart", "qend",
    "sstart", "send", "evalue", "bitscore"
]

hits = pd.read_csv(
    "search/uniref90_hits.tsv",
    sep="\t",
    names=cols
)

# qseqid is like "seq_0", "seq_1", ...
hits["idx"] = hits["qseqid"].str.replace("seq_", "").astype(int)

seq_lengths = df.set_index("idx")["sequence"].str.len().to_dict()

def compute_cov(row):
    qlen = seq_lengths.get(row["idx"], None)
    if qlen is None or qlen == 0:
        return np.nan
    return row["length"] / qlen

hits["coverage"] = hits.apply(compute_cov, axis=1)
hits_best = (
    hits
    .sort_values("bitscore", ascending=False)
    .groupby("idx")
    .head(1)
    .reset_index(drop=True)
)
PID_MIN = 40.0       # identity >= 40%
EVALUE_MAX = 1e-5    # e-value <= 1e-5
COV_MIN = 0.60       # coverage >= 60%

def is_reliable(row):
    if row["pident"] < PID_MIN:
        return False
    if row["evalue"] > EVALUE_MAX:
        return False
    if row["coverage"] < COV_MIN:
        return False
    return True

hits_best["reliable"] = hits_best.apply(is_reliable, axis=1)

print("[INFO] Seqs with any UniRef90 hit:", hits_best["idx"].nunique())
print("[INFO] Reliable hits:", hits_best["reliable"].sum())
df_ann = df.merge(hits_best, how="left", on="idx")

out_path = "clustered_sequences_with_uniref90.csv"
df_ann.to_csv(out_path, index=False)
print("[INFO] Saved annotated table to:", out_path)

df_ann.head()

# phylogenetic / homolog grouping

## UniRef90 Cluster

In [ ]:
import re
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("clustered_sequences_with_uniref90.csv")
df.head()

In [ ]:
# Checking for each UniRef90 ID, corrs to how many seqs
df_uniref = (
    df.groupby("sseqid")
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)

df_uniref.head(20)

In [ ]:
df["bio_family"] = df["sseqid"]

In [ ]:
df["is_far"] = df["pident"] < 80

In [ ]:
motif = re.compile(r"G.S.G")

def has_motif(seq):
    return bool(motif.search(seq))

df["has_motif"] = df["sequence"].apply(has_motif)

In [ ]:
# save a csv
df = pd.read_csv("clustered_sequences_with_uniref90.csv")

# New biological classification fields
df["bio_family"] = df["sseqid"]
df["mutation_level"] = 1 - df["pident"] / 100.0

# Select useful columns to save
cols = [
    "idx", "sequence", "cluster",
    "bio_family", "pident", "mutation_level",
    "umap_x", "umap_y"
]

bio_df = df[cols]
bio_df.head()

In [ ]:
bio_df.to_csv("bio_family_classification.csv", index=False)
print("[INFO] Saved to bio_family_classification.csv")


In [ ]:

# count size per family
family_counts = bio_df["bio_family"].value_counts()

# label large families separately, others merged
LARGE_FAMILY_MIN = 50
large_families = family_counts[family_counts >= LARGE_FAMILY_MIN].index.tolist()

bio_df["family_vis"] = bio_df["bio_family"].apply(
    lambda x: x if x in large_families else "Others"
)

plt.figure(figsize=(10, 9))
sns.scatterplot(
    data=bio_df,
    x="umap_x",
    y="umap_y",
    hue="family_vis",
    s=12,
    alpha=0.8,
    palette="tab20"
)
plt.title("UMAP colored by UniRef90 families (large families highlighted)")
plt.legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("umap_bio_family.png", dpi=300)
plt.show()

print("[INFO] Saved UMAP figure to umap_bio_family.png")


In [ ]:
top_fam = family_counts.head(20)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_fam.values, y=top_fam.index, palette="viridis")
plt.xlabel("Number of sequences")
plt.ylabel("UniRef90 family")
plt.title("Top 20 UniRef90 families (by count)")
plt.tight_layout()
plt.savefig("family_size_top20.png", dpi=300)
plt.show()

print("[INFO] Saved figure to family_size_top20.png")


In [ ]:
df_large = bio_df[bio_df["family_vis"] != "Others"]

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_large,
    x="family_vis",
    y="mutation_level",
    palette="Set2"
)
plt.xticks(rotation=90)
plt.title("Mutation Level by Biological Family")
plt.tight_layout()
plt.savefig("mutation_level_by_family.png", dpi=300)
plt.show()

print("[INFO] Saved figure to mutation_level_by_family.png")


## taxonomy

In [ ]:
import requests
import time

In [ ]:
df = pd.read_csv("clustered_sequences_with_uniref90.csv")

In [ ]:
df = df[df["sseqid"].notna()].copy()

In [ ]:
uniref_ids = sorted(df["sseqid"].unique())
print("[INFO] unique UniRef90 IDs:", len(uniref_ids))
uniref_ids[:5]

In [ ]:
def uniref_to_uniprot(uniref_id: str) -> str | None:
    """
    Convert UniRef90 ID like 'UniRef90_A0A1M7II12'
    to UniProtKB accession: 'A0A1M7II12'.
    """
    if not isinstance(uniref_id, str):
        return None
    parts = uniref_id.split("_")
    if len(parts) >= 2:
        return parts[1]
    return None

uniprot_ids = [uniref_to_uniprot(u) for u in uniref_ids]
print(uniprot_ids[:5])


In [ ]:
BASE_URL = "https://rest.uniprot.org/uniprotkb/{}"

def fetch_uniprot_taxonomy(acc: str, sleep_sec: float = 0.2):
    """
    Get taxonomy info for a UniProtKB accession via REST API.
    Return dict: {accession, organism_name, taxon_id, lineage_str}
    """
    url = BASE_URL.format(acc)
    try:
        r = requests.get(url, timeout=30)
        if r.status_code != 200:
            print(f"[WARN] {acc}: HTTP {r.status_code}")
            return {
                "uniprot_acc": acc,
                "organism_name": None,
                "taxon_id": None,
                "lineage_str": None,
            }
        j = r.json()
    except Exception as e:
        print(f"[ERROR] {acc}: {e}")
        return {
            "uniprot_acc": acc,
            "organism_name": None,
            "taxon_id": None,
            "lineage_str": None,
        }

    org = j.get("organism", {}) or {}
    organism_name = org.get("scientificName")
    taxon_id = org.get("taxonId")

    # lineage is a list，etc ["Bacteria", "Proteobacteria", ...]
    lineage = org.get("lineage", []) or []
    lineage_str = ";".join(lineage) if lineage else None

    # 
    if sleep_sec > 0:
        time.sleep(sleep_sec)

    return {
        "uniprot_acc": acc,
        "organism_name": organism_name,
        "taxon_id": taxon_id,
        "lineage_str": lineage_str,
    }


In [ ]:
tax_records = []
for i, (uniref_id, acc) in enumerate(zip(uniref_ids, uniprot_ids), start=1):
    if acc is None:
        continue
    info = fetch_uniprot_taxonomy(acc)
    info["sseqid"] = uniref_id   
    tax_records.append(info)

    if i % 50 == 0:
        print(f"[INFO] processed {i}/{len(uniref_ids)} UniRef90 IDs")

tax_df = pd.DataFrame(tax_records)
print("[INFO] taxonomy rows:", len(tax_df))
tax_df.head()


In [ ]:
tax_df.to_csv("uniref90_taxonomy.csv", index=False)
print("[INFO] Saved taxonomy table to uniref90_taxonomy.csv")

In [ ]:
tax_cols = ["sseqid", "uniprot_acc", "organism_name", "taxon_id", "lineage_str"]

df_tax = tax_df[tax_cols].drop_duplicates(subset=["sseqid"])

df_merged = df.merge(df_tax, how="left", on="sseqid")

out_path = "clustered_sequences_with_uniref90_taxonomy.csv"
df_merged.to_csv(out_path, index=False)
print("[INFO] Saved merged table to:", out_path)

df_merged[["cluster", "sseqid", "uniprot_acc", "organism_name", "taxon_id"]].head()


In [ ]:
###########

In [ ]:
df = pd.read_csv("clustered_sequences_with_uniref90_taxonomy.csv")
print(df.columns)
df[["sseqid", "organism_name", "lineage_str"]].head()

In [ ]:
from typing import Optional, Dict, List

def parse_lineage(lineage_str: Optional[str]) -> Dict[str, Optional[str]]:
    """
    Parse a lineage string like:
      'Bacteria;Proteobacteria;Gammaproteobacteria;Alteromonadales;Pseudoalteromonadaceae;Pseudoalteromonas'
    into a dict with keys:
      domain, phylum, class, order, family, genus
    If some levels are missing, they are set to None.
    """
    if not isinstance(lineage_str, str) or lineage_str.strip() == "":
        return {
            "domain": None,
            "phylum": None,
            "class": None,
            "order": None,
            "family": None,
            "genus": None,
        }

    parts: List[str] = [p.strip() for p in lineage_str.split(";") if p.strip()]

    # Pad to length >= 6
    while len(parts) < 6:
        parts.append(None)

    domain = parts[0] if len(parts) >= 1 else None
    phylum = parts[1] if len(parts) >= 2 else None
    klass  = parts[2] if len(parts) >= 3 else None
    order  = parts[3] if len(parts) >= 4 else None
    family = parts[4] if len(parts) >= 5 else None
    genus  = parts[5] if len(parts) >= 6 else None

    return {
        "domain": domain,
        "phylum": phylum,
        "class": klass,
        "order": order,
        "family": family,
        "genus": genus,
    }


In [ ]:
# Apply parse_lineage to each row and expand to columns
tax_expanded = df["lineage_str"].apply(lambda x: pd.Series(parse_lineage(x)))
df = pd.concat([df, tax_expanded], axis=1)

df[["sseqid", "organism_name", "domain", "phylum", "class", "order", "family", "genus"]].head()


In [ ]:
out_path = "clustered_sequences_with_uniref90_taxonomy_parsed.csv"
df.to_csv(out_path, index=False)
print("[INFO] Saved parsed taxonomy table to:", out_path)


In [ ]:
phylum_counts = df["phylum"].value_counts(dropna=True).to_frame(name="count")
phylum_counts.index.name = "phylum"
phylum_counts = phylum_counts.reset_index()

print(phylum_counts.head())
print(phylum_counts.columns)


In [ ]:
# Only keep top N phyla, others grouped as "Others"
TOP_N = 6
top_phyla = phylum_counts["phylum"].head(TOP_N).tolist()

def phylum_label(p):
    if p in top_phyla:
        return p
    else:
        return "Others"

df["phylum_vis"] = df["phylum"].apply(phylum_label)

plt.figure(figsize=(10, 9))
sns.scatterplot(
    data=df,
    x="umap_x",
    y="umap_y",
    hue="phylum_vis",
    s=12,
    alpha=0.8,
)
plt.title("UMAP colored by phylum")
plt.legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("umap_by_phylum.png", dpi=300)
plt.show()

print("[INFO] Saved UMAP-by-phylum plot to umap_by_phylum.png")


## UniRef90 parent x sequence diff 



In [ ]:
import pandas as pd
from Bio import SeqIO
from Bio import pairwise2
from tqdm.auto import tqdm

In [ ]:
# 1) Load main table (with sseqid, sequence)
df = pd.read_csv("clustered_sequences_with_uniref90_taxonomy_parsed.csv")

# Keep only rows that have UniRef90 ID
df = df[df["sseqid"].notna()].copy()
print("[INFO] sequences with UniRef90 hit:", len(df))

# 2) Load UniRef90 sequences into a dict: {UniRef90_ID -> sequence}
uniref_fasta_path = "search/uniref90.fasta"  # change if needed

uniref_dict = {}
for rec in SeqIO.parse(uniref_fasta_path, "fasta"):
    # rec.id should look like: 'UniRef90_A0A1M7II12'
    uniref_dict[rec.id] = str(rec.seq)

print("[INFO] loaded UniRef90 entries from FASTA:", len(uniref_dict))


In [ ]:
def seq_diff_vs_uniref90(parent_seq: str, child_seq: str):
    """
    Compute simple diff statistics between UniRef90 parent sequence and child sequence.
    Return dict with:
      parent_len, child_len, n_substitutions, n_insertions, n_deletions, alignment_identity
    """
    if parent_seq is None or child_seq is None:
        return {
            "parent_len": None,
            "seq_len": None,
            "n_substitutions": None,
            "n_insertions": None,
            "n_deletions": None,
            "alignment_identity": None,
        }

    # Global alignment; scoring scheme can be tuned if needed
    # match=2, mismatch=-1, gap_open=-5, gap_extend=-1
    alignments = pairwise2.align.globalms(child_seq, parent_seq, 2, -1, -5, -1, one_alignment_only=True)
    aln_child, aln_parent, score, start, end = alignments[0]

    subs = ins = dels = matches = 0

    for c_res, p_res in zip(aln_child, aln_parent):
        if c_res == "-" and p_res == "-":
            # shouldn't happen in a normal global alignment, but just in case
            continue
        elif c_res == "-" and p_res != "-":
            # deletion in child (relative to parent)
            dels += 1
        elif c_res != "-" and p_res == "-":
            # insertion in child
            ins += 1
        else:
            # both are amino acids
            if c_res == p_res:
                matches += 1
            else:
                subs += 1

    aln_len = matches + subs + ins + dels
    identity = matches / aln_len if aln_len > 0 else None

    return {
        "parent_len": len(parent_seq),
        "seq_len": len(child_seq),
        "n_substitutions": subs,
        "n_insertions": ins,
        "n_deletions": dels,
        "alignment_identity": identity,
    }


In [ ]:
# Choose how many sequences to process (None for all)
N_TEST = 200  # set to None later for all

if N_TEST is not None:
    df_sub = df.sample(n=min(N_TEST, len(df)), random_state=0).copy()
    print(f"[INFO] testing on {len(df_sub)} sequences")
else:
    df_sub = df.copy()
    print(f"[INFO] processing ALL sequences: {len(df_sub)}")

diff_results = []

for idx, row in tqdm(df_sub.iterrows(), total=len(df_sub)):
    uniref_id = row["sseqid"]  # e.g. 'UniRef90_A0A1M7II12'
    parent_seq = uniref_dict.get(uniref_id, None)
    child_seq = str(row["sequence"]).strip()

    stats = seq_diff_vs_uniref90(parent_seq, child_seq)
    stats["row_index"] = idx
    diff_results.append(stats)

diff_df = pd.DataFrame(diff_results).set_index("row_index")

# Merge back into df_sub
df_sub_diff = df_sub.join(diff_df, how="left")

df_sub_diff.head()


# Compare experimental reference sequences

## Cluster 

In [ ]:
EXCEL_PATH = Path("PETase.xlsx")
SHEETS = ["activity", "yield"]

In [ ]:
AA_RE = re.compile(r"^[ACDEFGHIKLMNPQRSTVWY]+$")

In [ ]:
def clean_seq(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    s = re.sub(r"\s+", "", s)
    if len(s) == 0 or (not AA_RE.match(s)):
        return np.nan
    return s

In [ ]:
def load_excel_activity_yield(excel_path):
    df_activity = pd.read_excel(excel_path, sheet_name="activity")
    df_yield    = pd.read_excel(excel_path, sheet_name="yield")

    # activity sheet
    act = df_activity.copy()
    act["sequence"] = act["Sequence"].map(clean_seq)
    act["protein_name"] = act.get("Name of protein ", act.get("Name of protein", np.nan))
    act["activity_val"] = act.get("Yield ", act.get("Yield", np.nan))  # The workbook labels this value column Yield (activity/yield-like values).
    act = act.dropna(subset=["sequence"])

    # yield sheet
    yld = df_yield.copy()
    yld["sequence"] = yld["Sequence"].map(clean_seq)
    yld["protein_name"] = yld.get("Name of protein ", yld.get("Name of protein", np.nan))
    yld["expr_yield"] = yld.get("yield (mg/ml)", yld.get("Yield", np.nan))
    yld = yld.dropna(subset=["sequence"])

    act_agg = (
        act.groupby("sequence", as_index=False)
        .agg(
            protein_name_activity=("protein_name", lambda s: s.dropna().astype(str).unique().tolist()),
            activity_mean=("activity_val", "mean"),
            activity_max=("activity_val", "max"),
            activity_n=("activity_val", "count"),
        )
    )
    yld_agg = (
        yld.groupby("sequence", as_index=False)
        .agg(
            protein_name_yield=("protein_name", lambda s: s.dropna().astype(str).unique().tolist()),
            expr_yield_mean=("expr_yield", "mean"),
            expr_yield_max=("expr_yield", "max"),
            expr_n=("expr_yield", "count"),
        )
    )

    out = act_agg.merge(yld_agg, on="sequence", how="outer")
    out["has_activity"] = out["activity_n"].fillna(0).astype(int) > 0
    out["has_yield"]    = out["expr_n"].fillna(0).astype(int) > 0
    return out


In [ ]:
excel_df = load_excel_activity_yield(EXCEL_PATH)
print("[INFO] Excel unique sequences:", len(excel_df))
display(excel_df.head())

In [ ]:
def esm_embed_like_before(sequences, model, batch_converter, device, batch_size=16, repr_layer=12):
    model.eval()
    all_embeddings = []
    t0 = time.time()

    num_seqs = len(sequences)
    for start in range(0, num_seqs, batch_size):
        end = min(start + batch_size, num_seqs)
        batch_seqs = sequences[start:end]

        batch_data = [(f"seq_{i}", s) for i, s in enumerate(batch_seqs, start=start)]
        labels_batch, strs, tokens = batch_converter(batch_data)
        tokens = tokens.to(device)

        with torch.no_grad():
            out = model(tokens, repr_layers=[repr_layer], return_contacts=False)
            rep = out["representations"][repr_layer]  # (B, L, D)

        emb = rep[:, 0, :].cpu().numpy() 
        all_embeddings.append(emb)

        if (start // batch_size) % 20 == 0:
            print(f"[INFO] Embedded {end}/{num_seqs} sequences in {time.time()-t0:.1f}s")

    return np.vstack(all_embeddings)

excel_seqs = excel_df["sequence"].tolist()
excel_emb = esm_embed_like_before(
    sequences=excel_seqs,
    model=model,
    batch_converter=batch_converter,
    device=device,
    batch_size=batch_size,
    repr_layer=12, 
)

print("[INFO] excel_emb shape:", excel_emb.shape)

In [ ]:
scaler_tmp = StandardScaler().fit(embeddings)

excel_scaled = scaler_tmp.transform(excel_emb)
excel_umap = reducer.transform(excel_scaled)

excel_df["umap_x"] = excel_umap[:, 0]
excel_df["umap_y"] = excel_umap[:, 1]

display(excel_df[["sequence","umap_x","umap_y","activity_max","expr_yield_max"]].head())

In [ ]:
# old space
X_old = X_umap
y_old = labels

# kNN
K = 15 
nn = NearestNeighbors(n_neighbors=K, metric="euclidean")
nn.fit(X_old)

dist, idx = nn.kneighbors(excel_umap, return_distance=True)
nbr_labels = y_old[idx]  # (N_new, K)

def vote_cluster(row_labels):
    row = row_labels[row_labels != -1]
    if row.size == 0:
        return -1, 0.0
    vals, counts = np.unique(row, return_counts=True)
    best = vals[counts.argmax()]
    strength = counts.max() / len(row_labels)   
    return int(best), float(strength)

pred = [vote_cluster(nbr_labels[i]) for i in range(len(excel_df))]
excel_df["pred_cluster_knn"] = [p[0] for p in pred]
excel_df["pred_strength_knn"] = [p[1] for p in pred]
excel_df["nn_dist_mean"] = dist.mean(axis=1)
excel_df["nn_dist_min"]  = dist.min(axis=1)

print(excel_df["pred_cluster_knn"].value_counts().head(20))
display(excel_df.sort_values(["pred_strength_knn","nn_dist_min"], ascending=[False, True]).head(20))

In [ ]:
# Cluster sizes in the original dataset.
old_cluster_counts = pd.Series(labels).value_counts().rename("old_cluster_size")

# Cluster assignment statistics for the workbook sequences.
excel_cluster_counts = excel_df["pred_cluster_knn"].value_counts().rename("excel_seq_count")

excel_func = (
    excel_df.groupby("pred_cluster_knn", as_index=True)
    .agg(
        excel_n=("sequence","count"),
        has_activity=("has_activity","sum"),
        has_yield=("has_yield","sum"),
        activity_top=("activity_max","max"),
        activity_mean=("activity_mean","mean"),
        expr_yield_top=("expr_yield_max","max"),
        expr_yield_mean=("expr_yield_mean","mean"),
        mean_strength=("pred_strength_knn","mean"),
        mean_nn_dist=("nn_dist_mean","mean"),
    )
)

summary = pd.concat([old_cluster_counts, excel_cluster_counts, excel_func], axis=1).fillna(0)
summary = summary.sort_values(["excel_n","activity_top","expr_yield_top"], ascending=False)

display(summary.head(30))


In [ ]:
x_old = X_umap[:, 0]
y_old = X_umap[:, 1]
lab_old = labels

# new
x_new = excel_df["umap_x"].to_numpy()
y_new = excel_df["umap_y"].to_numpy()

plt.figure(figsize=(9,7))
sc = plt.scatter(x_old, y_old, c=lab_old, s=6, alpha=0.35)
plt.scatter(x_new, y_new, s=70, facecolors="none", edgecolors="k", linewidths=1.2, alpha=0.9)

plt.title("UMAP clusters (old) + Excel sequences (overlay)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.colorbar(sc, label="Old cluster label")
plt.show()

## Check seq

In [ ]:
orig_seq_col = None
for c in df.columns:
    if c.lower() in ["sequence", "seq", "protein_sequence"]:
        orig_seq_col = c
        break

if orig_seq_col is None:
    raise ValueError("Cannot find sequence column in original df")

print("[INFO] original sequence column:", orig_seq_col)
print("[INFO] #original sequences:", df[orig_seq_col].notna().sum())
print("[INFO] #excel sequences:", excel_df['sequence'].notna().sum())

In [ ]:
orig_seqs = set(df[orig_seq_col].dropna().astype(str))
excel_seqs = set(excel_df["sequence"].dropna().astype(str))

shared_seqs = orig_seqs.intersection(excel_seqs)

print(f"[INFO] exact sequence matches: {len(shared_seqs)}")


In [ ]:
orig_seq_col = None
for c in df.columns:
    if c.lower() in ["sequence", "seq", "protein_sequence"]:
        orig_seq_col = c
        break
if orig_seq_col is None:
    raise ValueError("Cannot find sequence column in original df")

orig_seqs = (
    df[[orig_seq_col]]
    .dropna()
    .drop_duplicates()
    .rename(columns={orig_seq_col: "sequence"})
    .reset_index(drop=True)
)


excel_seqs = (
    excel_df[["sequence"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

print("[INFO] #orig sequences :", len(orig_seqs))
print("[INFO] #excel sequences:", len(excel_seqs))


def global_identity(seq1, seq2):
    aln = pairwise2.align.globalxx(seq1, seq2, one_alignment_only=True, score_only=False)
    if len(aln) == 0:
        return 0.0
    a1, a2, score, start, end = aln[0]
    matches = sum((c1 == c2) for c1, c2 in zip(a1, a2) if c1 != "-" and c2 != "-")
    aligned_len = max(len(seq1), len(seq2))
    return matches / aligned_len


In [ ]:
SIM_TH = 0.90

hits = []

for i, row_e in excel_seqs.iterrows():
    seq_e = row_e["sequence"]
    len_e = len(seq_e)

    for j, row_o in orig_seqs.iterrows():
        seq_o = row_o["sequence"]

        if abs(len(seq_o) - len_e) > 0.1 * len_e:
            continue

        ident = global_identity(seq_e, seq_o)
        if ident >= SIM_TH:
            hits.append({
                "excel_sequence": seq_e,
                "orig_sequence": seq_o,
                "identity": ident,
            })

print(f"[INFO] #pairs with identity >= {SIM_TH*100:.0f}% :", len(hits))


In [ ]:
def global_identity_and_mismatches(seq1, seq2):
    """
    globalxx alignment:
    - identity = matches / max(len1, len2)
    - mismatches = max(len1, len2) - matches   
    """
    aln = pairwise2.align.globalxx(seq1, seq2, one_alignment_only=True, score_only=False)
    if len(aln) == 0:
        return 0.0, None
    a1, a2, score, start, end = aln[0]

    matches = 0
    for c1, c2 in zip(a1, a2):
        if c1 == "-" or c2 == "-":
            continue
        if c1 == c2:
            matches += 1

    denom = max(len(seq1), len(seq2))
    identity = matches / denom if denom > 0 else 0.0
    mismatches = denom - matches
    return identity, mismatches

SIM_TH = 0.95
hits_99 = []

for i, row_e in excel_seqs.iterrows():
    seq_e = row_e["sequence"]
    len_e = len(seq_e)
    max_mis = math.floor((1 - SIM_TH) * max(len_e, 1))  # 99% -> ~1% mismatches allowed

    for j, row_o in orig_seqs.iterrows():
        seq_o = row_o["sequence"]


        if abs(len(seq_o) - len_e) > 0.02 * len_e:  
            continue


        if abs(len(seq_o) - len_e) > max_mis:
            continue

        ident, mism = global_identity_and_mismatches(seq_e, seq_o)
        if ident >= SIM_TH:
            hits_99.append({
                "excel_sequence": seq_e,
                "orig_sequence": seq_o,
                "identity": ident,
                "mismatches_est": mism,
            })

print(f"[INFO] #pairs with identity >= {SIM_TH*100:.0f}% :", len(hits_99))


In [ ]:
def kmers(s, k=5):
    return {s[i:i+k] for i in range(0, max(len(s)-k+1, 0))}

def jaccard(a, b):
    if not a and not b:
        return 1.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union else 0.0

def global_identity(seq1, seq2):
    aln = pairwise2.align.globalxx(seq1, seq2, one_alignment_only=True, score_only=False)
    if len(aln) == 0:
        return 0.0
    a1, a2, score, start, end = aln[0]
    matches = sum((c1 == c2) for c1, c2 in zip(a1, a2) if c1 != "-" and c2 != "-")
    denom = max(len(seq1), len(seq2))
    return matches / denom if denom else 0.0

In [ ]:
KMER_K = 5
orig_list = orig_seqs["sequence"].tolist()
orig_kmers = [kmers(s, k=KMER_K) for s in orig_list]

print("[INFO] precomputed kmers for orig:", len(orig_kmers))


In [ ]:
TOPK_CAND = 200     # Number of candidates retained by the k-mer prefilter; larger values increase cost.
LEN_TOL = 0.10      # Restrict reference candidates to lengths within 10% of the query.

results = []

excel_list = excel_seqs["sequence"].tolist()
print("[INFO] excel sequences:", len(excel_list))

for idx_e, seq_e in enumerate(excel_list):
    len_e = len(seq_e)
    km_e = kmers(seq_e, k=KMER_K)

    # 1) Filter candidates by length.
    cand_idx = [i for i, s in enumerate(orig_list) if abs(len(s) - len_e) <= LEN_TOL * len_e]
    if not cand_idx:
        # Use all references if the length filter removes every candidate.
        cand_idx = list(range(len(orig_list)))

    # 2) Prefilter by k-mer Jaccard similarity.
    sims = []
    for i in cand_idx:
        sims.append((jaccard(km_e, orig_kmers[i]), i))
    sims.sort(reverse=True, key=lambda x: x[0])
    top_cand = [i for _, i in sims[:TOPK_CAND]]

    # 3) Compute global alignment identity.
    best = (-1.0, None)  # (identity, orig_index)
    top5 = []
    for i in top_cand:
        ident = global_identity(seq_e, orig_list[i])
        top5.append((ident, i))
        if ident > best[0]:
            best = (ident, i)

    top5.sort(reverse=True, key=lambda x: x[0])
    best_ident, best_i = best

    results.append({
        "excel_idx": idx_e,
        "excel_len": len_e,
        "best_identity": best_ident,
        "best_orig_sequence": orig_list[best_i] if best_i is not None else None,
        "best_orig_len": len(orig_list[best_i]) if best_i is not None else None,
        "top5_identities": [round(x[0], 4) for x in top5[:5]],
        "top5_orig_indices": [x[1] for x in top5[:5]],
    })

    if (idx_e + 1) % 5 == 0 or (idx_e + 1) == len(excel_list):
        print(f"[INFO] processed {idx_e+1}/{len(excel_list)}")

besthit_df = pd.DataFrame(results).sort_values("best_identity", ascending=False)
display(besthit_df.head(22))


In [ ]:
# Detect the cluster column.
cluster_col = None
for c in df.columns:
    if c.lower() in ["cluster", "label", "labels", "cluster_id"]:
        cluster_col = c
        break
print("[INFO] cluster column:", cluster_col)

# Build a deduplicated lookup table for the original sequences.
orig_lookup = df[[orig_seq_col] + ([cluster_col] if cluster_col else [])].drop_duplicates(subset=[orig_seq_col])
orig_lookup = orig_lookup.rename(columns={orig_seq_col: "best_orig_sequence"})

# merge excel metadata
excel_meta = excel_df.copy()
excel_meta = excel_meta.rename(columns={"sequence": "excel_sequence"})
besthit_df2 = besthit_df.merge(excel_meta, left_on="excel_idx", right_index=True, how="left")

# merge orig cluster info
besthit_df2 = besthit_df2.merge(orig_lookup, on="best_orig_sequence", how="left")

# Select readable output columns.
cols = [
    "excel_idx", "best_identity", "excel_len", "best_orig_len",
    "pred_cluster_knn" if "pred_cluster_knn" in besthit_df2.columns else None,
    cluster_col,
    "activity_max", "expr_yield_max",
    "protein_name_activity", "protein_name_yield",
    "top5_identities",
]
cols = [c for c in cols if c is not None and c in besthit_df2.columns]

display(besthit_df2[cols].head(22))


In [ ]:
besthit_df2.to_csv("excel_best_hit_in_original_by_identity.csv", index=False)
print("[INFO] saved excel_best_hit_in_original_by_identity.csv")


## compare

In [ ]:
WT_PATH = "pet-2025-wildtype-cds.csv"

wt_df = pd.read_csv(WT_PATH)

# Select the protein sequence column explicitly.
wt_df["sequence"] = (
    wt_df["Wt AA Sequence"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Discard empty or invalid sequences.
wt_df = wt_df[wt_df["sequence"].str.match(r"^[ACDEFGHIKLMNPQRSTVWY]+$")]
wt_df = wt_df.drop_duplicates(subset=["sequence"]).reset_index(drop=True)

print("[INFO] wildtype protein sequences:", len(wt_df))
display(wt_df[["sequence"]].head())

In [ ]:
wt_df["sequence"]

In [ ]:
# Excel（activity / yield）
excel_seqs = (
    excel_df[["sequence"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
print("[INFO] excel sequences:", len(excel_seqs))

# Original clustered sequence set.
orig_seq_col = None
for c in df.columns:
    if c.lower() in ["sequence", "seq", "protein_sequence"]:
        orig_seq_col = c
        break
if orig_seq_col is None:
    raise ValueError("Cannot find sequence column in original df")

orig_seqs = (
    df[[orig_seq_col]]
    .dropna()
    .drop_duplicates()
    .rename(columns={orig_seq_col: "sequence"})
    .reset_index(drop=True)
)
print("[INFO] original (4800+) sequences:", len(orig_seqs))


In [ ]:
wt_set    = set(wt_df["sequence"])
excel_set = set(excel_seqs["sequence"])
orig_set  = set(orig_seqs["sequence"])

print("Exact matches:")
print("  WT ∩ Excel :", len(wt_set & excel_set))
print("  WT ∩ Orig  :", len(wt_set & orig_set))
print("  Excel ∩ Orig:", len(excel_set & orig_set))
print("  WT ∩ Excel ∩ Orig:", len(wt_set & excel_set & orig_set))


In [ ]:
def global_identity(seq1, seq2):
    aln = pairwise2.align.globalxx(seq1, seq2, one_alignment_only=True)
    if len(aln) == 0:
        return 0.0
    a1, a2, score, start, end = aln[0]
    matches = sum(
        (c1 == c2) for c1, c2 in zip(a1, a2)
        if c1 != "-" and c2 != "-"
    )
    return matches / max(len(seq1), len(seq2))

In [ ]:
results = []

excel_list = excel_seqs["sequence"].tolist()
orig_list  = orig_seqs["sequence"].tolist()

for wt_seq in wt_df["sequence"]:
    # Excel best
    best_excel = (-1.0, None)
    for ex in excel_list:
        if abs(len(ex) - len(wt_seq)) > 0.2 * len(wt_seq):
            continue
        ident = global_identity(wt_seq, ex)
        if ident > best_excel[0]:
            best_excel = (ident, ex)

    # Orig best
    best_orig = (-1.0, None)
    for og in orig_list:
        if abs(len(og) - len(wt_seq)) > 0.2 * len(wt_seq):
            continue
        ident = global_identity(wt_seq, og)
        if ident > best_orig[0]:
            best_orig = (ident, og)

    results.append({
        "wildtype_sequence": wt_seq,
        "wt_len": len(wt_seq),
        "best_excel_identity": best_excel[0],
        "best_orig_identity": best_orig[0],
        "best_excel_sequence": best_excel[1],
        "best_orig_sequence": best_orig[1],
    })

wt_compare_df = pd.DataFrame(results).sort_values(
    ["best_excel_identity", "best_orig_identity"], ascending=False
)

display(wt_compare_df.head(20))


In [ ]:
# Cluster column.
cluster_col = None
for c in df.columns:
    if c.lower() in ["cluster", "label", "labels", "cluster_id"]:
        cluster_col = c
        break

orig_lookup = (
    df[[orig_seq_col] + ([cluster_col] if cluster_col else [])]
    .drop_duplicates(subset=[orig_seq_col])
    .rename(columns={orig_seq_col: "best_orig_sequence"})
)

excel_lookup = (
    excel_df
    .rename(columns={"sequence": "best_excel_sequence"})
)

wt_compare_df = (
    wt_compare_df
    .merge(orig_lookup, on="best_orig_sequence", how="left")
    .merge(excel_lookup, on="best_excel_sequence", how="left", suffixes=("_orig", "_excel"))
)

display(
    wt_compare_df[
        [
            "wt_len",
            "best_excel_identity",
            "best_orig_identity",
            cluster_col if cluster_col else None,
            "activity_max",
            "expr_yield_max",
        ]
    ].head(20)
)
